In [0]:
query = """
    SELECT
        ROW_NUMBER() OVER (ORDER BY pr.start_date, pr.product_key) AS product_number,
        pr.product_id,
        pr.product_key,
        pr.product_name,
        pr.cat_id,
        pc.category,
        pc.subcategory,
        pc.maintenance_flag,
        pr.product_line,
        pr.start_date
    FROM silver.crm_products pr
    LEFT JOIN silver.erp_product_category pc
    ON pr.cat_id = pc.category_id
"""
df = spark.sql(query)

In [0]:
df.limit(30).display()

In [0]:
total_count = df.count()
distinct_count = df.select("product_id").distinct().count()
if total_count > distinct_count:
    print(f"Warning: {total_count - distinct_count} duplicate records found in the DataFrame.")
else:
    print(f"All records in the DataFrame are unique.")

# Writing Gold Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("workspace.gold.dim_products")

In [0]:
%sql
Select * from workspace.gold.dim_products limit 10